# Python and pandas foundations

[Run in browser](https://muzammilafroz.github.io/applied-economics-data-learning-lab/lab/index.html?path=lessons/01_python_pandas_foundations.ipynb) | [Open in Colab](https://colab.research.google.com/github/muzammilafroz/applied-economics-data-learning-lab/blob/main/notebooks/lessons/01_python_pandas_foundations.ipynb) | [Course home](https://muzammilafroz.github.io/applied-economics-data-learning-lab/) | [Take the test](https://muzammilafroz.github.io/applied-economics-data-learning-lab/tests/?module=module-01)

> This independent learning resource uses fictional, synthetic data. It is not an official assessment or credential.

## Why this lesson matters

Applied data work begins before any model or map. You need to know what an object contains, how rows and columns are selected, what missing values mean, and whether a proposed key is actually unique.

Prerequisites: none. You will learn imports, variables, types, indexing, masks, functions, assertions, and safe joins.

In [1]:
from __future__ import annotations

import os
import sys
import types
from urllib.request import urlopen

if sys.platform == "emscripten":
    import piplite
    await piplite.install("pyodide-http")
    import pyodide_http
    pyodide_http.patch_all()

RAW_CODE_ROOT = "https://raw.githubusercontent.com/muzammilafroz/applied-economics-data-learning-lab/v1.0.1/learning_lab"
if sys.platform == "emscripten":
    from js import window
    if window.location.hostname in {"127.0.0.1", "localhost"}:
        RAW_CODE_ROOT = f"{window.location.origin}/learning_lab"
        os.environ["LEARNING_LAB_DATA_BASE"] = f"{window.location.origin}/data/teaching"

def load_public_module(module_name):
    """Import locally, or fetch the small public helper when running in Colab/Lite."""
    try:
        return __import__(f"learning_lab.{module_name}", fromlist=[module_name])
    except ModuleNotFoundError:
        location = f"{RAW_CODE_ROOT}/{module_name}.py"
        source = urlopen(location).read().decode("utf-8")
        module = types.ModuleType(f"learning_lab.{module_name}")
        exec(compile(source, location, "exec"), module.__dict__)
        return module

lab_io = load_public_module("io")
get_data_url = lab_io.get_data_url
read_teaching_csv = lab_io.read_teaching_csv
read_teaching_geojson = lab_io.read_teaching_geojson

print("Runtime:", sys.platform)
print("Data reference:", os.getenv("LEARNING_LAB_DATA_REF", "v1.0.1"))

Runtime: win32
Data reference: v1.0.1


In [2]:
import pandas as pd

production = read_teaching_csv("district_production.csv")
points = read_teaching_csv("survey_points.csv")

print("production shape:", production.shape)
print("points shape:", points.shape)
display(production.head())

production shape: (25, 2)
points shape: (80, 4)


,d,production
0,1,188
1,2,175
2,3,430
3,4,661
4,5,855


## Objects, names, and types

`production` is a variable. It points to a pandas `DataFrame`, which is a rectangular table. `shape` returns a tuple: `(number_of_rows, number_of_columns)`. `dtypes` reports how pandas represents each column.

An integer stores a whole number. A floating-point value can store a decimal approximation. A string stores text. A Boolean is `True` or `False`.

In [3]:
print(type(production))
print(production.dtypes)

district_codes = production["d"]
print("One selected column is a", type(district_codes).__name__)
print("First three codes:", district_codes.iloc[:3].tolist())

<class 'pandas.core.frame.DataFrame'>
d             int64
production    int64
dtype: object
One selected column is a Series
First three codes: [1, 2, 3]


## Indexing and masks

Square brackets select a column. `.loc[row_rule, columns]` selects by labels and a Boolean rule. `.iloc` selects by numerical position. A mask is a sequence of `True` and `False` values that tells pandas which rows to keep.

In [4]:
high_production = production["production"] > production["production"].median()
selected = production.loc[high_production, ["d", "production"]]

print("Rows above the median:", len(selected))
display(selected.head())

Rows above the median: 12


,d,production
13,14,7180
14,15,8202
15,16,9390
16,17,10701
17,18,12070


## Nullable values and explicit checks

Missing is not the same as zero. `isna()` identifies missing values. An assertion turns an assumption into an executable contract. If the condition is false, Python stops with an `AssertionError` near the source of the problem.

In [5]:
missing_by_column = production.isna().sum()
print(missing_by_column)

assert production["d"].notna().all(), "district code is missing"
assert production["d"].is_unique, "district code must be unique"
assert production["production"].ge(0).all(), "production cannot be negative"
print("All district-table contracts passed.")

d             0
production    0
dtype: int64
All district-table contracts passed.


## Labels are not always keys

The GeoJSON deliberately contains two districts named `Riverbend`. A display name helps a reader, but it may be duplicated or edited. The code `d` is the stable, unique key.

In [6]:
districts = read_teaching_geojson()
duplicate_names = districts.loc[
    districts["district_name"].duplicated(keep=False),
    ["d", "district_name"],
]
display(duplicate_names)

assert districts["d"].is_unique
assert not districts["district_name"].is_unique

,d,district_name
6,7,Riverbend
17,18,Riverbend


## A safe join

`merge` combines tables. `on="d"` names the shared key. `how="left"` keeps every district geometry. `validate="one_to_one"` requires each key to appear at most once on both sides. `indicator=True` records whether each row matched.

In [7]:
district_data = districts.merge(
    production,
    on="d",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print(district_data["_merge"].value_counts())
assert district_data["production"].notna().all()
district_data = district_data.drop(columns="_merge")

_merge
both          25
left_only      0
right_only     0
Name: count, dtype: int64


## Functions package a rule

A function gives a meaningful name to reusable steps. Parameters are the inputs named in the definition. `return` sends a result back to the caller. Type hints document expected types but do not automatically enforce them.

In [8]:
def count_rows_for_group(frame: pd.DataFrame, group: str) -> int:
    # Return the number of rows whose group exactly matches group.
    matches = frame["group"].eq(group)
    return int(matches.sum())

group_counts = {group: count_rows_for_group(points, group) for group in sorted(points["group"].unique())}
print(group_counts)
assert sum(group_counts.values()) == len(points)

{'A': 50, 'B': 15, 'C': 10, 'D': 5}


## Common failure: letting pandas guess an identifier

An identifier can look numeric while behaving like text. Arithmetic on a household ID is meaningless, and leading zeros can matter. Read such columns with `dtype={"hhid": "string"}`. This choice preserves the original representation before validation.

What would break if we skipped this? A value such as `04000001` could become `4000001`, and joins to a text key could fail.

In [9]:
survey_preview = read_teaching_csv(
    "lumen_2018_survey.csv",
    dtype={"hhid": "string"},
    nrows=3,
)
print(survey_preview["hhid"].tolist())
print("Stripped lengths:", survey_preview["hhid"].str.strip().str.len().tolist())

['   04000000', '   04000001', '   04000002']
Stripped lengths: [8, 8, 8]


## Guided practice

Try these before the test:

1. Use `value_counts()` on the point group column.
2. Use `nunique()` to count district codes.
3. Change `validate="one_to_one"` to a deliberately wrong key in a copy of the code and read the error.

Self-check: Why is a failing assertion useful? It stops the workflow before an invalid assumption silently changes later results.

Next: open the module test. Its final cell creates a JSON answer bundle that can be pasted into the website form.